# [Notebook]Unihack26_Full_Pixel_Audit
Research preparation only. Original source splits preserved. No accuracy or production checkpoint produced. Parcel3D renders stay academic-only. Group: Unihack26.

In [ ]:
from pathlib import Path
import sys,json
SOURCES={'ml/__init__.py': '', 'ml/data/__init__.py': '', 'ml/data/parcel3d_audit.py': '"""CPU-only audit of a mounted Parcel3D input. Does not copy its image archive."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport platform\nfrom collections import Counter, deque\nfrom datetime import UTC, datetime\nfrom pathlib import Path, PurePosixPath\n\nfrom PIL import Image\n\nSOURCE = "ayesha19765/parcel3d"\nORIGINAL = "https://zenodo.org/records/8032204"\nALLOWED_SPLITS = ("train", "validation", "test")\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef within_root(root: Path, relative: str) -> Path:\n    relative = relative.replace("\\\\", "/")\n    parsed = PurePosixPath(relative)\n    if parsed.is_absolute() or ".." in parsed.parts or ":" in relative:\n        raise ValueError("UNSAFE_ANNOTATION_PATH")\n    result = (root / Path(*parsed.parts)).resolve()\n    if not result.is_relative_to(root.resolve()):\n        raise ValueError("PATH_OUTSIDE_INPUT")\n    return result\n\n\ndef find_annotations(input_root: Path) -> dict[str, Path]:\n    # Kaggle can expose both legacy and owner-qualified aliases of one input.\n    # Annotation files sit above per-image directories; never walk 13,400 samples\n    # just to locate three files. Bound discovery and compare duplicate contents.\n    roots, queue, visited = [], deque([(input_root, 0)]), set()\n    while queue:\n        directory, depth = queue.popleft()\n        resolved = directory.resolve()\n        if resolved in visited:\n            continue\n        visited.add(resolved)\n        if len(visited) > 200:\n            raise RuntimeError("ANNOTATION_DISCOVERY_LIMIT")\n        if all((directory / f"parcel3d_{split}.json").is_file() for split in ALLOWED_SPLITS):\n            roots.append(directory)\n            continue\n        if depth < 5:\n            for child in sorted(directory.iterdir()):\n                if (\n                    child.name not in {*ALLOWED_SPLITS, "misc"}\n                    and not child.name.startswith(".")\n                    and child.is_dir()\n                ):\n                    queue.append((child, depth + 1))\n    if not roots:\n        raise RuntimeError("ANNOTATION_ROOT_NOT_FOUND")\n    roots.sort(key=lambda path: (len(path.parts), str(path)))\n    if len(roots) > 1:\n        digests = {\n            tuple(sha256_file(root / f"parcel3d_{split}.json") for split in ALLOWED_SPLITS)\n            for root in roots\n        }\n        if len(digests) != 1:\n            raise RuntimeError("ANNOTATION_ALIASES_HAVE_DIFFERENT_CONTENT")\n    print(f"Annotation roots: {len(roots)} equivalent mount(s); selected {roots[0]}", flush=True)\n    return {split: roots[0] / f"parcel3d_{split}.json" for split in ALLOWED_SPLITS}\n\n\ndef resolve_image(annotation_path: Path, image_record: dict, split: str) -> Path | None:\n    filename = image_record.get("file_name")\n    if not isinstance(filename, str):\n        return None\n    candidates = [\n        within_root(annotation_path.parent, filename),\n        within_root(annotation_path.parent, f"{split}/{filename}"),\n    ]\n    for candidate in candidates:\n        if candidate.is_file():\n            return candidate\n    return None\n\n\ndef audit(input_root: Path, output_root: Path, source_version: int = 1) -> dict:\n    input_root = input_root.resolve()\n    output_root = output_root.resolve()\n    if output_root.is_relative_to(input_root):\n        raise ValueError("OUTPUT_MUST_NOT_BE_INSIDE_READ_ONLY_INPUT")\n    output_root.mkdir(parents=True, exist_ok=True)\n    print("Locating Parcel3D annotation roots (bounded discovery)", flush=True)\n    annotations = find_annotations(input_root)\n    result = {\n        "schema_version": "parcel3d-audit-v2",\n        "source": SOURCE,\n        "source_version": source_version,\n        "original_record": ORIGINAL,\n        "checked_at": datetime.now(UTC).isoformat(),\n        "runtime": {"python": platform.python_version(), "platform": platform.platform()},\n        "source_kind": "SYNTHETIC",\n        "splits": {},\n        "errors": [],\n        "license": {\n            "mirror_metadata": "Apache-2.0; not authoritative for upstream assets",\n            "original_full_dataset": "Academic research only; Zenodo other-nc",\n            "textureless_obj_meshes": "CC-BY-4.0, separate grant in original record",\n            "cardboard_textures": "CC-BY-4.0, separate record 8041823",\n            "training_policy": "Full rendered-image training remains gated pending scope review.",\n            "commercial_release": "NOT_CLEARED_FOR_ORIGINAL_RENDERED_IMAGES",\n            "reference_matching_labels": "Must inspect; COCO damage labels alone do not establish cross-view correspondence or evidence sufficiency.",\n        },\n    }\n    for package in ("numpy", "Pillow", "torch", "opencv-python"):\n        try:\n            result["runtime"][package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            pass\n    group_sets = {}\n    sample_directories = set()\n    for split, path in annotations.items():\n        print(f"Auditing {split}: {path.name}", flush=True)\n        document = json.loads(path.read_text(encoding="utf-8"))\n        images = document.get("images", [])\n        labels = document.get("annotations", [])\n        record = {\n            "annotation_path": str(path.relative_to(input_root)),\n            "annotation_sha256": sha256_file(path),\n            "annotation_bytes": path.stat().st_size,\n            "top_level_keys": sorted(document),\n            "image_count": len(images),\n            "annotation_count": len(labels),\n            "categories": document.get("categories", []),\n            "image_fields": dict(Counter(k for image in images for k in image)),\n            "annotation_fields": dict(Counter(k for annotation in labels for k in annotation)),\n            "sample_records": images[:2],\n            "sample_annotations": labels[:2],\n            "samples": [],\n        }\n        ids = [image.get("id") for image in images]\n        record["duplicate_image_ids"] = len(ids) - len(set(ids))\n        group_key = next(\n            (\n                key\n                for key in ("mesh_id", "object_id", "parcel_id", "model_id")\n                if images and all(key in item for item in images)\n            ),\n            None,\n        )\n        record["explicit_group_key"] = group_key\n        record["group_split_verified"] = False\n        if group_key:\n            group_sets[split] = {str(item[group_key]) for item in images}\n        # Deterministic stratified positions; no exhaustive image read or copy.\n        positions = sorted({int(i * max(len(images) - 1, 0) / 7) for i in range(8)})\n        for index in positions if images else []:\n            image_record = images[index]\n            image_path = resolve_image(path, image_record, split)\n            sample = {\n                "image_id": image_record.get("id"),\n                "file_name": image_record.get("file_name"),\n            }\n            if image_path is None:\n                sample["status"] = "PATH_UNRESOLVED"\n                result["errors"].append(\n                    {"split": split, "image_id": sample["image_id"], "code": "PATH_UNRESOLVED"}\n                )\n            else:\n                try:\n                    with Image.open(image_path) as image:\n                        image.load()\n                        sample.update(\n                            status="READ_OK",\n                            width=image.width,\n                            height=image.height,\n                            format=image.format,\n                            mode=image.mode,\n                        )\n                    sample["sha256"] = sha256_file(image_path)\n                    sample["bytes"] = image_path.stat().st_size\n                    sample["relative_path"] = str(image_path.relative_to(input_root))\n                    sample_directories.add(image_path.parent)\n                except (OSError, ValueError) as exc:\n                    sample.update(status="READ_FAILED", error_type=type(exc).__name__)\n                    result["errors"].append(\n                        {"split": split, "image_id": sample["image_id"], "code": "READ_FAILED"}\n                    )\n            record["samples"].append(sample)\n        result["splits"][split] = record\n    root = annotations["train"].parent\n    extensions = Counter()\n    mesh_examples = []\n    metadata_examples = []\n    # Inventory only sampled image neighborhoods. Do not imply a complete count.\n    print(f"Inspecting metadata/meshes near {len(sample_directories)} sampled images", flush=True)\n    for sample_directory in sorted(sample_directories):\n        for directory, _, filenames in os.walk(sample_directory):\n            for filename in sorted(filenames):\n                suffix = Path(filename).suffix.lower()\n                extensions[suffix] += 1\n                path = Path(directory) / filename\n                if suffix == ".obj" and len(mesh_examples) < 8:\n                    mesh_examples.append(\n                        {\n                            "path": str(path.relative_to(root)),\n                            "bytes": path.stat().st_size,\n                            "sha256": sha256_file(path),\n                        }\n                    )\n                if (\n                    suffix == ".json"\n                    and path not in annotations.values()\n                    and len(metadata_examples) < 8\n                ):\n                    meta = {"path": str(path.relative_to(root)), "bytes": path.stat().st_size}\n                    if path.stat().st_size < 100_000:\n                        try:\n                            meta["document"] = json.loads(path.read_text(encoding="utf-8"))\n                        except (ValueError, UnicodeError):\n                            meta["status"] = "UNREADABLE_JSON"\n                    metadata_examples.append(meta)\n    result["inventory"] = {\n        "scope": "SAMPLED_IMAGE_NEIGHBORHOODS",\n        "sample_directory_count": len(sample_directories),\n        "sampled_extension_counts": dict(extensions),\n        "mesh_examples": mesh_examples,\n        "metadata_examples": metadata_examples,\n    }\n    if len(group_sets) == 3:\n        overlaps = {\n            f"{a}:{b}": len(group_sets[a] & group_sets[b])\n            for a, b in (("train", "validation"), ("train", "test"), ("validation", "test"))\n        }\n        result["group_overlap_counts"] = overlaps\n        for record in result["splits"].values():\n            record["group_split_verified"] = not any(overlaps.values())\n    result["mount_read_gate"] = (\n        "PASS"\n        if not result["errors"]\n        and all(\n            any(s["status"] == "READ_OK" for s in record["samples"])\n            for record in result["splits"].values()\n        )\n        else "FAIL"\n    )\n    result["correspondence_label_gate"] = "PENDING_REVIEW"\n    result["commercial_license_gate"] = "HOLD_ORIGINAL_RENDERS"\n    result["full_archive_copied"] = False\n    target = output_root / "parcel3d-audit.json"\n    target.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")\n    (output_root / "audit-sha256.txt").write_text(\n        sha256_file(target) + "  parcel3d-audit.json\\n", encoding="ascii"\n    )\n    print(\n        json.dumps(\n            {\n                "mount_read_gate": result["mount_read_gate"],\n                "counts": {k: v["image_count"] for k, v in result["splits"].items()},\n                "outputs": str(output_root),\n            },\n            ensure_ascii=False,\n        )\n    )\n    return result\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input", type=Path, default=Path("/kaggle/input"))\n    parser.add_argument("--output", type=Path, default=Path("/kaggle/working/unihack26-audit"))\n    parser.add_argument("--source-version", type=int, default=1)\n    args = parser.parse_args()\n    audit(args.input, args.output, args.source_version)\n', 'ml/data/upgrade_audit.py': '"""Full Parcel3D pixel/duplicate audit on Kaggle, without exporting source images."""\nfrom __future__ import annotations\n\nimport hashlib\nimport io\nimport json\nfrom collections import Counter, defaultdict\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nfrom PIL import Image\n\nfrom ml.data.parcel3d_audit import find_annotations, within_root\n\nEXPECTED_INDEX_SHA = "55cf3a02babbe1e902b7524ec39d0a7195b2a4f4b2a1ac001a066668e4600a6d"\n\n\ndef assign_roles(records):\n    """Keep official train/test, partition official validation by object identity."""\n    splits = defaultdict(set)\n    for row in records:\n        splits[row["lineage_id"]].add(row["source_split"])\n    if any(len(values) != 1 for values in splits.values()):\n        raise ValueError("OFFICIAL_SOURCE_LINEAGE_LEAKAGE")\n    validation = sorted(\n        (key for key, values in splits.items() if values == {"validation"}),\n        key=lambda key: hashlib.sha256(f"kc-upgrade-v2:{key}".encode()).digest(),\n    )\n    if len(validation) < 2:\n        raise ValueError("NEED_TWO_VALIDATION_LINEAGES")\n    selection = set(validation[:len(validation) // 2])\n    return {\n        key: ("selection" if key in selection else "calibration")\n        if values == {"validation"} else next(iter(values))\n        for key, values in splits.items()\n    }\n\n\ndef duplicate_groups(rows, field):\n    groups = defaultdict(list)\n    for row in rows:\n        groups[row[field]].append(row)\n    return [{"digest": digest, "paths": [r["image_path"] for r in group],\n             "roles": sorted({r["role"] for r in group})}\n            for digest, group in groups.items() if len(group) > 1]\n\n\ndef run(input_root, output_root, index_path, workers=4):\n    output_root = Path(output_root)\n    output_root.mkdir(parents=True, exist_ok=True)\n    raw = Path(index_path).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != EXPECTED_INDEX_SHA:\n        raise ValueError("LINEAGE_INDEX_HASH_MISMATCH")\n    records = [json.loads(line) for line in raw.splitlines()]\n    roles = assign_roles(records)\n    root = find_annotations(Path(input_root))["train"].parent\n\n    def inspect(row):\n        path = within_root(root, row["image_path"])\n        raw_image = path.read_bytes()\n        with Image.open(io.BytesIO(raw_image)) as source:\n            rgb = source.convert("RGB")\n            pixel_hash = hashlib.sha256(\n                f"RGB:{rgb.width}:{rgb.height}:".encode() + rgb.tobytes(),\n            ).hexdigest()\n            # This fingerprint is for later near-duplicate review, not a proof\n            # that perceptually similar images are independent.\n            thumb = rgb.convert("L").resize((9, 8))\n            pixels = list(thumb.getdata())\n            bits = [pixels[y * 9 + x] > pixels[y * 9 + x + 1] for y in range(8) for x in range(8)]\n            dhash = sum(int(bit) << index for index, bit in enumerate(bits))\n            return {"image_path": row["image_path"], "lineage_id": row["lineage_id"],\n                    "source_split": row["source_split"], "role": roles[row["lineage_id"]],\n                    "width": rgb.width, "height": rgb.height, "bytes": len(raw_image),\n                    "sha256": hashlib.sha256(raw_image).hexdigest(), "pixel_sha256": pixel_hash,\n                    "dhash": f"{dhash:016x}", "category_id": row["category_id"],\n                    "parcel_type": row["parcel_type"]}\n\n    output = []\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        for index, result in enumerate(pool.map(inspect, records)):\n            output.append(result)\n            if (index + 1) % 500 == 0:\n                print(json.dumps({"decoded": index + 1, "total": len(records)}), flush=True)\n    byte_duplicates = duplicate_groups(output, "sha256")\n    pixel_duplicates = duplicate_groups(output, "pixel_sha256")\n    leakage = [group for group in byte_duplicates + pixel_duplicates if len(group["roles"]) > 1]\n    expected = {Path(row["image_path"]).parts[1] for row in records}\n    actual = {p.name for split in ("train", "validation", "test") for p in (root / split).iterdir() if p.is_dir()}\n    payload = "".join(json.dumps(row, sort_keys=True) + "\\n" for row in output)\n    (output_root / "images.jsonl").write_text(payload, encoding="utf-8")\n    report = {"status": "PASS" if not leakage else "HOLD_DUPLICATE_LEAKAGE",\n              "images": len(output), "lineages": len(roles), "roles": dict(Counter(r["role"] for r in output)),\n              "lineage_roles": dict(Counter(roles.values())), "bytes_read": sum(r["bytes"] for r in output),\n              "index_sha256": EXPECTED_INDEX_SHA, "manifest_sha256": hashlib.sha256(payload.encode()).hexdigest(),\n              "byte_duplicate_groups": byte_duplicates, "decoded_duplicate_groups": pixel_duplicates,\n              "cross_role_duplicate_groups": leakage, "unannotated_directories": sorted(actual - expected),\n              "missing_directories": sorted(expected - actual),\n              "unannotated_policy": "EXCLUDED_NO_LABEL_INFERENCE", "near_duplicate_review": "PENDING",\n              "pair_labels": "NOT_CREATED", "physical_box_validation": "NOT_RUN",\n              "license": "ACADEMIC_RESEARCH_ONLY_UPSTREAM_RENDER_TERMS"}\n    (output_root / "audit.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    print(json.dumps({k: v for k, v in report.items() if not isinstance(v, list)}), flush=True)\n    return report\n'}
root=Path('/kaggle/working/_src')
for name,source in SOURCES.items():
 p=root/name
 p.parent.mkdir(parents=True,exist_ok=True)
 p.write_text(source,encoding='utf-8')
sys.path.insert(0,str(root))


In [ ]:
from ml.data.upgrade_audit import run
from collections import deque
queue = deque([(Path('/kaggle/input'), 0)])
found = set()
while queue:
    directory, depth = queue.popleft()
    candidate = directory / 'lineage-index.jsonl'
    if candidate.is_file():
        found.add(candidate.resolve())
    if depth < 6:
        for child in directory.iterdir():
            if child.is_dir() and child.name not in {'parcel3d','train','validation','test','misc'}:
                queue.append((child, depth + 1))
assert found, 'LINEAGE_NOTEBOOK_INPUT_MISSING'
result = run('/kaggle/input', '/kaggle/working/audit', sorted(found)[0])
assert result['status'] == 'PASS', result['status']
Path('/kaggle/working/source-hashes.json').write_text(json.dumps({'ml/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/parcel3d_audit.py': '08ddf5f683f5b23c8364228d53f8b3883518465c3e2d66a64bf3ff9888f46b4e', 'ml/data/upgrade_audit.py': '0eb325d6339412d847aba9ce0ea1b0e45486ac589987c798efbabdff7bb77b66'},indent=2))
